# Face recognition with deepface and Weaviate

[deepface](https://github.com/serengil/deepface) is a lightweight face recognition and facial attribute analysis library for Python. Besides comparing images directly, it can **register** face embeddings in a database and then **search** and **identify** faces against them. Weaviate is one of its database backends.

This notebook runs deepface's Weaviate integration end to end against a local Weaviate:

| Step | What it shows |
|---|---|
| 1. Register | face embeddings stored in a Weaviate collection, duplicates skipped |
| 2. Search | 1:N search, exact scan and HNSW (`ann`) giving the same matches |
| 3. Identify | 1:1 verification against a registered face |
| 4. Inside Weaviate | the collection deepface creates and what each object holds |
| 5. Connection options | self-hosted, Weaviate Cloud, auth, timeouts |
| 6. Multi-tenancy | one isolated gallery of faces per tenant |
| 7. Quantization | compressed vectors with unchanged distances |
| 8. What reached Weaviate | no GraphQL, and the `X-Weaviate-Client-Integration` header |

Every step checks its result with `assert`, so running the whole notebook is also a test of the integration.

> **Note:** the Weaviate v4 integration (gRPC, no GraphQL, tenants and quantization) is not in a deepface release yet. Install the modified library as shown in the setup cell below.

**Versions used:** Weaviate 1.39.7 · weaviate-client 4.23.1 · deepface 0.0.102 with the v4 integration · Facenet + OpenCV detector

**Author:** Duda Nogueira

## Setup

Start Weaviate from this folder:

```sh
docker compose up -d
```

It exposes REST on `8080`, gRPC on `50051`, the debug endpoints on `6060` (to read the integration header) and Prometheus metrics on `2112` (to count GraphQL requests). If a port is taken, set `WEAVIATE_HTTP_PORT`, `WEAVIATE_GRPC_PORT`, `WEAVIATE_DEBUG_PORT` or `WEAVIATE_METRICS_PORT` for both `docker compose` and this notebook.

Then install the dependencies. deepface needs Python 3.9 to 3.12 and a backend engine, TensorFlow here.

In [1]:
# ---------------------------------------------------------------------------
# Install the modified deepface library (Weaviate v4 integration, not released yet).
# Uncomment ONE of the options below and run this cell once.
#
# Option 1: from a local checkout of the branch
#   git clone https://github.com/serengil/deepface.git
#   cd deepface
#   git remote add duda https://github.com/dudanogueira/deepface.git
#   git fetch duda feat/weaviate-quantization-tenancy
#   git checkout -b feat/weaviate-quantization-tenancy duda/feat/weaviate-quantization-tenancy
# then point pip at the checkout:
# %pip install -q -e /path/to/deepface
#
# Option 2: straight from the branch, without a checkout
# %pip install -q "deepface @ git+https://github.com/dudanogueira/deepface.git@feat/weaviate-quantization-tenancy"
#
# Once a deepface release includes the integration, use instead:
# %pip install -q deepface
# ---------------------------------------------------------------------------

%pip install -q -r requirements.txt

In [2]:
import json
import os
import re
import urllib.request
import warnings
from collections import Counter
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import pandas as pd
import weaviate
from deepface import DeepFace
from deepface.modules.database import weaviate as deepface_weaviate

HTTP_PORT = int(os.environ.get("WEAVIATE_HTTP_PORT", "8080"))
GRPC_PORT = int(os.environ.get("WEAVIATE_GRPC_PORT", "50051"))
DEBUG_PORT = int(os.environ.get("WEAVIATE_DEBUG_PORT", "6060"))
METRICS_PORT = int(os.environ.get("WEAVIATE_METRICS_PORT", "2112"))

assert hasattr(deepface_weaviate, "INTEGRATION_NAME"), (
    "This deepface build still uses the Weaviate v3 client. Install the modified library"
    " with the setup cell above, then restart the kernel."
)

# what deepface receives as connection_details: a url, a dict of options, or a JSON string
CONNECTION = {"url": f"http://localhost:{HTTP_PORT}", "grpc_port": GRPC_PORT}

# an admin client, used by this notebook only to look inside Weaviate
client = weaviate.connect_to_local(port=HTTP_PORT, grpc_port=GRPC_PORT)
print(f"Weaviate {client.get_meta()['version']} · weaviate-client {weaviate.__version__}"
      f" · {deepface_weaviate.INTEGRATION_NAME}")

Weaviate 1.39.7 · weaviate-client 4.23.1 · deepface/0.0.102


### Sample faces and helpers

The images come from deepface's own test dataset. `img1`, `img2`, `img4`, `img5`, `img6`, `img7`, `img10` and `img11` show the same person, the others show different people. `img1` is the probe we search with, and is never registered.

In [3]:
DATASET_URL = "https://raw.githubusercontent.com/serengil/deepface/master/tests/unit/dataset"
IMAGES_DIR = Path("images")

PROBE = "img1.jpg"
GALLERY = ["img2.jpg", "img3.jpg", "img4.jpg", "img5.jpg", "img9.jpg", "img10.jpg", "img12.jpg"]
SAME_PERSON = {"img2.jpg", "img4.jpg", "img5.jpg", "img10.jpg"}

# model and detector used throughout; deepface stores each model/detector/alignment/
# normalization combination in its own Weaviate collection
MODEL = {"model_name": "Facenet", "detector_backend": "opencv"}
COLLECTIONS = {
    "plain": "Embeddings_facenet_opencv_aligned_raw",
    "tenants": "Embeddings_facenet_opencv_unaligned_raw",
    "quantized": "Embeddings_facenet_opencv_aligned_norm",
}


def image(name):
    """Download a dataset image once, and return its local path."""
    path = IMAGES_DIR / name
    if not path.exists():
        IMAGES_DIR.mkdir(exist_ok=True)
        urllib.request.urlretrieve(f"{DATASET_URL}/{name}", path)
    return str(path)


def names(df):
    return [Path(n).name for n in df["img_name"]]


def graphql_requests():
    """Requests to /v1/graphql so far, from Weaviate's Prometheus metrics."""
    text = urllib.request.urlopen(f"http://localhost:{METRICS_PORT}/metrics").read().decode()
    pattern = r'weaviate_http_request_duration_seconds_count\{.*route="/v1/graphql".*\} (\S+)'
    return sum(float(m) for m in re.findall(pattern, text))


def integration_requests():
    """Requests received with X-Weaviate-Client-Integration: deepface/<version>."""
    url = f"http://localhost:{DEBUG_PORT}/debug/telemetry/clients"
    usage = json.load(urllib.request.urlopen(url)).get("clientIntegrationUsage", {})
    return sum(usage.get("deepface", {}).values())


# start from a clean state, and remember the request counters
for collection in COLLECTIONS.values():
    client.collections.delete(collection)
graphql_before, header_before = graphql_requests(), integration_requests()

## 1. Register faces

`DeepFace.register` detects the faces in an image, computes one embedding per face and stores it with `database_type="weaviate"`. deepface creates the collection on first use. Each face is stored once: registering the same face again inserts nothing, and `inserted` reports what was actually written.

In [4]:
inserted = 0
for name in GALLERY:
    result = DeepFace.register(
        img=image(name), database_type="weaviate", connection_details=CONNECTION, **MODEL
    )
    inserted += result["inserted"]
print(f"registered {inserted} faces from {len(GALLERY)} images")
assert inserted == len(GALLERY)

again = DeepFace.register(
    img=image(GALLERY[0]), database_type="weaviate", connection_details=CONNECTION, **MODEL
)
print("registering", GALLERY[0], "again inserted:", again["inserted"])
assert again["inserted"] == 0

registered 7 faces from 7 images
registering img2.jpg again inserted: 0


## 2. Search

`DeepFace.search` is the database-backed version of `DeepFace.find`, a 1:N search. It returns one DataFrame per face found in the probe image, with the registered faces within the model's distance threshold.

- `search_method="exact"` reads every stored embedding and compares them all in deepface.
- `search_method="ann"` sends the probe embedding to Weaviate, which finds the nearest faces with its HNSW index.

Both return the same faces and distances.

In [5]:
search = {**MODEL, "distance_metric": "euclidean", "database_type": "weaviate", "connection_details": CONNECTION}

exact = DeepFace.search(img=image(PROBE), search_method="exact", **search)[0]
ann = DeepFace.search(img=image(PROBE), search_method="ann", **search)[0]

assert set(names(exact)) == SAME_PERSON
assert names(ann) == names(exact)
assert all(abs(a - e) < 1e-4 for a, e in zip(ann["distance"], exact["distance"]))

ann.assign(img_name=names(ann))[["img_name", "distance", "threshold", "confidence", "id"]]

,img_name,distance,threshold,confidence,id
0,img10.jpg,7.433843,10,69.14,0938e47c-3aaa-511d-a98c-a597e1a0028d
1,img4.jpg,7.735756,10,65.28,a99eb56d-abbc-5c0d-9aba-c5afd510e6da
2,img5.jpg,7.919453,10,62.93,da86ccd4-61a0-5858-b749-846122e2b2c7
3,img2.jpg,8.037711,10,61.42,4019141a-53f1-5e7b-bf21-b40f246d239d


## 3. Identify

`DeepFace.identify` is the database-backed version of `DeepFace.verify`, a 1:1 check. It compares an image with one registered face, looked up by its Weaviate object id.

In [6]:
best_match = ann.iloc[0]
verdict = DeepFace.identify(
    img=image(PROBE),
    identity_id=best_match["id"],
    distance_metric="euclidean",
    database_type="weaviate",
    connection_details=CONNECTION,
    **MODEL,
)
print(f"{PROBE} vs {Path(best_match['img_name']).name}: verified={verdict['verified']},"
      f" distance={verdict['distance']:.3f}, threshold={verdict['threshold']}")
assert verdict["verified"]

img1.jpg vs img10.jpg: verified=True, distance=7.434, threshold=10.0


## 4. Inside Weaviate

deepface created the collection with its own vectors (no vectorizer), L2-squared distance for raw embeddings and cosine distance for L2-normalized ones. Each object stores the face embedding as its vector, plus the metadata deepface needs. The vector is 128-dimensional for Facenet.

In [7]:
faces = client.collections.use(COLLECTIONS["plain"])
config = faces.config.get()
print("collection:", config.name)
print("distance:  ", config.vector_index_config.distance_metric.value)
print("objects:   ", len(faces))

obj = faces.query.fetch_objects(limit=1, include_vector=True).objects[0]
print("vector:    ", len(obj.vector["default"]), "dimensions")
{k: v for k, v in obj.properties.items() if k not in ("face", "embedding")}

collection: Embeddings_facenet_opencv_aligned_raw
distance:   l2-squared
objects:    7
vector:     128 dimensions


{'l2_normalized': False,
 'embedding_hash': 'd1f2d4f0ddc6a160ec139968ba09f6a6306797eb8fdd1cbbd0734fb51717ebc3',
 'detector_backend': 'opencv',
 'face_shape': [160, 160, 3],
 'face_hash': 'bc16d8c193593db6938e943eae98466ec3dcb7225d27181a6d658e7a289446d5',
 'model_name': 'Facenet',
 'aligned': True,
 'img_name': 'images/img10.jpg'}

## 5. Connection options

`connection_details` takes a URL, a dict or the same dict as a JSON string. The JSON form is how the deepface API server receives it, through `DEEPFACE_CONNECTION_DETAILS`. The dict keys follow the weaviate client's `connect_to_*` helpers. The deployment type is inferred from the URL:

```python
# self-hosted, gRPC on the default port 50051
connection_details = "http://localhost:8080"

# Weaviate Cloud
connection_details = {"url": "https://my-cluster.weaviate.cloud", "api_key": "..."}

# custom deployment: separate gRPC host, OIDC auth, timeouts
connection_details = {
    "url": "https://weaviate.example.com",
    "grpc_host": "grpc.weaviate.example.com",
    "grpc_port": 443,
    "auth": {"client_secret": "...", "scope": "openid"},  # or api_key, access_token, username/password
    "timeout": {"init": 5, "query": 60, "insert": 120},
    "skip_init_checks": False,
    "headers": {"X-Custom-Header": "..."},
}
```

Other options are `deployment` (`custom`, `cloud` or `local`), `http_host`, `http_port`, `http_secure`, `grpc_secure`, `proxies`, `trust_env`, `connection_config`, `grpc_config` and `additional_config`. If an option is not given, the environment variables `DEEPFACE_WEAVIATE_URI`, `WEAVIATE_API_KEY`, `DEEPFACE_WEAVIATE_GRPC_PORT`, `DEEPFACE_WEAVIATE_TIMEOUT` and `DEEPFACE_WEAVIATE_SKIP_INIT_CHECKS` are used. Unknown options raise an error, so typos don't pass silently:

In [8]:
try:
    DeepFace.search(img=image(PROBE), database_type="weaviate",
                    connection_details={**CONNECTION, "grpcport": 50051}, **MODEL)
except ValueError as e:
    print(e)

Unknown Weaviate connection option(s): ['grpcport']. Valid options are: ['additional_config', 'api_key', 'auth', 'connection_config', 'deployment', 'grpc_config', 'grpc_host', 'grpc_port', 'grpc_secure', 'headers', 'http_host', 'http_port', 'http_secure', 'proxies', 'quantization', 'skip_init_checks', 'tenant', 'timeout', 'trust_env', 'url']


You can also pass a `weaviate.WeaviateClient` you created yourself as `connection`. deepface then uses it as is and leaves closing it to you:

In [9]:
own = DeepFace.search(img=image(PROBE), search_method="ann", connection=client,
                      distance_metric="euclidean", database_type="weaviate", **MODEL)[0]
assert names(own) == names(ann)
names(own)

['img10.jpg', 'img4.jpg', 'img5.jpg', 'img2.jpg']

## 6. Multi-tenancy: one gallery per tenant

With a `tenant`, deepface keeps each gallery of faces isolated with [Weaviate multi-tenancy](https://docs.weaviate.io/weaviate/manage-collections/multi-tenancy). Use it when one Weaviate serves several customers, sites or events, or several deepface services. A search only sees faces registered with the same tenant. Tenants are created on their first register, and deleting a tenant erases its faces.

A collection is created as multi-tenant only when a tenant is given, and it keeps that mode, so this step uses another collection (unaligned faces). The tenant can also come from `DEEPFACE_WEAVIATE_TENANT`.

In [10]:
TENANTS = {"studio-a": ["img2.jpg", "img3.jpg"], "studio-b": ["img4.jpg", "img12.jpg"]}
tenant_model = {**MODEL, "align": False}

for tenant, gallery in TENANTS.items():
    for name in gallery:
        DeepFace.register(img=image(name), database_type="weaviate",
                          connection_details={**CONNECTION, "tenant": tenant}, **tenant_model)

for tenant, gallery in TENANTS.items():
    found = DeepFace.search(
        img=image(PROBE),
        search_method="ann",
        similarity_search=True,  # return every face in the gallery, not only the matches
        distance_metric="euclidean",
        database_type="weaviate",
        connection_details={**CONNECTION, "tenant": tenant},
        **tenant_model,
    )[0]
    print(f"{tenant} sees {names(found)}")
    assert sorted(names(found)) == sorted(gallery)

tenants = sorted(client.collections.use(COLLECTIONS["tenants"]).tenants.get())
print("tenants in Weaviate:", tenants)
assert tenants == sorted(TENANTS)

studio-a sees ['img2.jpg', 'img3.jpg']
studio-b sees ['img4.jpg', 'img12.jpg']
tenants in Weaviate: ['studio-a', 'studio-b']


A multi-tenant collection needs a tenant on every call. Leaving it out raises an error that says so:

In [11]:
try:
    DeepFace.search(img=image(PROBE), search_method="ann", database_type="weaviate",
                    connection_details=CONNECTION, **tenant_model)
    raise AssertionError("expected an error")
except ValueError as e:
    print(e)

Weaviate collection Embeddings_facenet_opencv_unaligned_raw has multi-tenancy enabled, so a tenant is required. Set the tenant option or DEEPFACE_WEAVIATE_TENANT.


## 7. Quantization

`quantization` compresses the vectors of new collections to cut memory use:

- `rq`: 8-bit rotational quantization, Weaviate 1.32+
- `rq-1`: 1-bit, Weaviate 1.33+
- `bq`, `sq`, `pq`
- `none`: opt out of a server default quantization, such as Weaviate Cloud's

Weaviate rescores the results with the original vectors, so distances, and therefore deepface's thresholds, are unchanged. Servers older than a quantizer would silently ignore it, so deepface checks the server version first. It can also be set with `DEEPFACE_WEAVIATE_QUANTIZATION`.

This step uses L2-normalized embeddings, which live in their own collection with cosine distance.

In [12]:
quantized = {**CONNECTION, "quantization": "rq"}
normalized = {**MODEL, "l2_normalize": True}

for name in GALLERY:
    DeepFace.register(img=image(name), database_type="weaviate", connection_details=quantized, **normalized)

quantizer = client.collections.use(COLLECTIONS["quantized"]).config.get().vector_index_config.quantizer
print("quantizer:", quantizer)
assert type(quantizer).__name__ == "_RQConfig"

q_search = {**normalized, "distance_metric": "cosine", "database_type": "weaviate", "connection_details": quantized}
q_exact = DeepFace.search(img=image(PROBE), search_method="exact", **q_search)[0]
q_ann = DeepFace.search(img=image(PROBE), search_method="ann", **q_search)[0]
assert names(q_ann) == names(q_exact)
assert all(abs(a - e) < 1e-4 for a, e in zip(q_ann["distance"], q_exact["distance"]))

pd.DataFrame({"img_name": names(q_ann), "ann distance": q_ann["distance"], "exact distance": q_exact["distance"]})

quantizer: _RQConfig(cache=None, bits=8, rescore_limit=20, centering=False, training_limit=10000)


,img_name,ann distance,exact distance
0,img10.jpg,0.198820,0.198819
1,img4.jpg,0.215827,0.215826
2,img2.jpg,0.235442,0.235442
3,img5.jpg,0.235483,0.235483


## 8. What reached Weaviate

The integration uses weaviate-client v4. Searches, full scans and batch inserts go over gRPC, and schema calls over REST. Nothing uses the GraphQL API, which older deepface versions called for every search and every registered face. Every request also carries `X-Weaviate-Client-Integration: deepface/<version>`, which Weaviate counts per integration.

In [13]:
graphql = graphql_requests() - graphql_before
header = integration_requests() - header_before
print(f"GraphQL requests: {int(graphql)}")
print(f"requests with the {deepface_weaviate.INTEGRATION_NAME} header: {header}")
assert graphql == 0
assert header > 0

GraphQL requests: 0
requests with the deepface/0.0.102 header: 177


## Clean up

In [14]:
for collection in COLLECTIONS.values():
    client.collections.delete(collection)
client.close()

Stop Weaviate and delete its data with `docker compose down -v`.